In [14]:
library(dplyr)
## visualization
library(ggplot2)
library(ggplot2)
library(gridExtra)
library(dplyr)
library(ggplot2)
library(cowplot)

In [15]:
setwd("/home/liuxuan/metadata/PRJEB12953")
initial_meta<-read.csv("SraRunTable.csv",header = T)
head(initial_meta)

,Run,Assay.Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center.Name,Consent,DATASTORE.filetype,⋯,LibrarySource,Organism,Platform,ReleaseDate,create_date,version,Sample.Name,sample_name,SRA.Study,Submitter_Id
,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>
1,ERR1303943,AMPLICON,295,21727417,PRJEB12953,SAMEA3888926,10923248,"SEOUL NATIONAL UNIVERSITY, KOREA",public,"fastq,run.zq,sra",⋯,METAGENOMIC,human oral metagenome,ILLUMINA,2018-01-05T00:00:00Z,2018-01-05T10:22:00Z,1,SAMEA3888926,P001,ERP014487,P001
2,ERR1303944,AMPLICON,295,18315137,PRJEB12953,SAMEA3888927,9323758,"SEOUL NATIONAL UNIVERSITY, KOREA",public,"fastq,run.zq,sra",⋯,METAGENOMIC,human oral metagenome,ILLUMINA,2018-01-05T00:00:00Z,2018-01-05T10:23:00Z,1,SAMEA3888927,P002,ERP014487,P002
3,ERR1303945,AMPLICON,295,16360723,PRJEB12953,SAMEA3888928,8421346,"SEOUL NATIONAL UNIVERSITY, KOREA",public,"fastq,run.zq,sra",⋯,METAGENOMIC,human oral metagenome,ILLUMINA,2018-01-05T00:00:00Z,2018-01-05T10:23:00Z,1,SAMEA3888928,P003,ERP014487,P003
4,ERR1303946,AMPLICON,294,11703414,PRJEB12953,SAMEA3888929,5958136,"SEOUL NATIONAL UNIVERSITY, KOREA",public,"fastq,run.zq,sra",⋯,METAGENOMIC,human oral metagenome,ILLUMINA,2018-01-05T00:00:00Z,2018-01-05T10:23:00Z,1,SAMEA3888929,P004,ERP014487,P004
5,ERR1303947,AMPLICON,295,11096210,PRJEB12953,SAMEA3888930,5608081,"SEOUL NATIONAL UNIVERSITY, KOREA",public,"fastq,run.zq,sra",⋯,METAGENOMIC,human oral metagenome,ILLUMINA,2018-01-05T00:00:00Z,2018-01-05T10:23:00Z,1,SAMEA3888930,P005,ERP014487,P005
6,ERR1303948,AMPLICON,295,9845654,PRJEB12953,SAMEA3888931,4961712,"SEOUL NATIONAL UNIVERSITY, KOREA",public,"fastq,run.zq,sra",⋯,METAGENOMIC,human oral metagenome,ILLUMINA,2018-01-05T00:00:00Z,2018-01-05T10:23:00Z,1,SAMEA3888931,P006,ERP014487,P006


In [16]:
colnames(initial_meta)  

[1] "Run"                    "Assay.Type"             "AvgSpotLen"            
 [4] "Bases"                  "BioProject"             "BioSample"             
 [7] "Bytes"                  "Center.Name"            "Consent"               
[10] "DATASTORE.filetype"     "DATASTORE.provider"     "DATASTORE.region"      
[13] "ENA.FIRST.PUBLIC..run." "ENA_first_public"       "ENA.LAST.UPDATE..run." 
[16] "ENA_last_update"        "Experiment"             "external_id"           
[19] "INSDC_center_alias"     "INSDC_center_name"      "INSDC_first_public"    
[22] "INSDC_last_update"      "INSDC_status"           "Instrument"            
[25] "Library.Name"           "LibraryLayout"          "LibrarySelection"      
[28] "LibrarySource"          "Organism"               "Platform"              
[31] "ReleaseDate"            "create_date"            "version"               
[34] "Sample.Name"            "sample_name"            "SRA.Study"             
[37] "Submitter_Id"

In [17]:
uniform_meta <- data.frame(
    BioProject = initial_meta$BioProject,
    Run = initial_meta$Run,
    sample_name = initial_meta$'sample_name',
    Age = NA,
    BMI = NA,
    Sex = NA,
    Region = "V4",
    Platform = initial_meta$Platform,
    Country = 'Korea',
    Site = 'subgingival plaque',
    public_time = initial_meta$ReleaseDate,
    PCR_primers = NA,
    title = 'Oral Microbiota: Microbial Biomarkers of Metabolic Syndrome Independent of Host Genetic Factors',
    Disease_study = "NO",
    Status = NA,
    matched_samples = 'NO'
)

In [18]:
print(length(unique(uniform_meta$Run)))

print(length(unique(uniform_meta$sample_name)))

[1] 228
[1] 228


In [19]:
plot_data <- uniform_meta %>%
  select(-BioProject, -Run, -sample_name, -title)

In [20]:
create_simple_plots <- function(data) {
  plots <- list()
  
  for (col in colnames(data)) {
    # Check if the column is all NA
    if (all(is.na(data[[col]]))) {
      # Create an empty plot with text indicating all NA
      p <- ggplot() +
        annotate("text", x = 0.5, y = 0.5, label = "this column is NA", size = 6) +
        labs(title = col) +
        theme_void()
    } else if (is.numeric(data[[col]])) {
      # Numeric variable: histogram
      p <- ggplot(data, aes(x = .data[[col]])) +
        geom_histogram(fill = "blue", alpha = 0.6, bins = 20, na.rm = TRUE) +
        labs(title = col) +
        theme_minimal()
    } else {
      # Categorical variable: bar plot (remove NA values)
      count_data <- as.data.frame(table(data[[col]], useNA = "ifany"))
      p <- ggplot(count_data, aes(x = reorder(Var1, -Freq), y = Freq)) +
        geom_col(fill = "red", alpha = 0.6) +
        labs(title = col, x = NULL, y = "Count") +
        theme_minimal() +
        theme(axis.text.x = element_text(angle = 15, hjust = 1))
    }
    plots[[col]] <- p
  }
  
  return(plots)
}

In [21]:
plots <- create_simple_plots(plot_data)
p<-plot_grid(plots$Age, 
          plots$BMI, 
          plots$Sex, 
          plots$Region, 
          plots$Platform, 
          plots$Country, 
          plots$Site,
          plots$public_time, 
          plots$PCR_primers,
          plots$Disease_study,
          plots$Status,
          plots$matched_samples,# add 
          ncol = 3, align = "hv", axis = "lb")
ggsave("Metadata_Summary.png", plot = p, width = 8, height = 6, dpi = 150)

In [22]:
## uniform the column order
uniform_meta <- uniform_meta[, c("BioProject", "Run", "sample_name", "Age", "BMI", "Sex", "Region", "Platform", "Country", "Site", "public_time", "PCR_primers", "title", "Disease_study", "Status")]
write.csv(uniform_meta, file = "/home/liuxuan/metadata/PRJEB12953/uniform_meta.csv", quote = F)